<a href="https://colab.research.google.com/github/adalves-ufabc/2026.Q3-PLN/blob/main/2026_Q3_PLN_AULA_16_Notebook_35.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Processamento de Linguagem Natural [2026-Q3]**
Prof. Alexandre Donizeti Alves

## **Agentes [LangChain]**
---

## 1. O que é um agente?

Um agente é um sistema no qual um modelo de linguagem decide quando utilizar ferramentas, quais argumentos fornecer e se precisa realizar novas chamadas antes de responder. O ciclo de execução termina quando o modelo considera que a tarefa foi concluída.

Na API atual, `create_agent` cria o agente a partir de um modelo, uma lista de ferramentas e instruções de sistema. O agente recebe uma sequência de mensagens e devolve um estado que inclui o histórico atualizado.

**Diferença importante:** uma cadeia tradicional executa uma sequência definida pelo desenvolvedor; um agente pode escolher dinamicamente quais ferramentas utilizar e em que ordem.

## 2. Instalação e configuração

Instale as integrações necessárias. Em uma sessão nova do Colab, execute esta célula uma vez.

In [ ]:
%pip install -qU langchain langchain-openai

In [ ]:
import langchain
import langchain_openai

print("LangChain:", langchain.__version__)
print("Integração OpenAI:", langchain_openai.__version__)

LangChain: 1.5.0
Integração OpenAI: 1.8.0


Configure a chave da API. O uso de `getpass` evita que a chave fique visível no código ou no histórico da saída.

In [ ]:
from getpass import getpass
import os

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Informe sua chave da API da OpenAI: ")

## 3. Inicialização do modelo

O modelo é instanciado separadamente e passado para `create_agent`. O identificador abaixo mantém o modelo utilizado no material original. Caso não esteja habilitado na sua conta, substitua-o por outro modelo compatível com a API da OpenAI e com chamadas de ferramentas.

In [ ]:
from langchain_openai import ChatOpenAI

modelo = ChatOpenAI( model="gpt-6-luna" )

## 4. Primeiro agente: uma ferramenta simples

Uma ferramenta é uma função Python disponibilizada ao modelo. O decorador `@tool` registra a função, enquanto a *docstring* descreve ao modelo quando ela deve ser utilizada. Os nomes e as descrições das ferramentas influenciam a escolha feita pelo agente.

In [ ]:
from langchain.tools import tool
from langchain.agents import create_agent

@tool
def converter_celsius_fahrenheit(celsius: float) -> float:
    """Converte uma temperatura de graus Celsius para Fahrenheit."""
    return (celsius * 9 / 5) + 32

agente_temperatura = create_agent(
    model = modelo,
    tools = [converter_celsius_fahrenheit],
    system_prompt = "Você é um assistente didático. Explique os cálculos de forma concisa."
)

Faça uma pergunta cuja resposta exija a ferramenta. Observe que o usuário não precisa solicitar explicitamente a chamada da função: o agente decide utilizá-la.

In [ ]:
resposta = agente_temperatura.invoke({
    "messages": [
        {"role": "user", "content": "Quantos graus Fahrenheit correspondem a 25 graus Celsius?"}
    ]
})

print(resposta["messages"][-1].content)

[{'type': 'text', 'text': '25 °C correspondem a **77 °F**. A conversão é \\(F = \\frac{9}{5}C + 32\\): \\(\\frac{9}{5}\\times25 + 32 = 77\\).', 'annotations': [], 'id': 'msg_07b7f8516bcc88e9006afb1f56e1b087d298e3f6010c1a62bf', 'phase': 'final_answer'}]


In [ ]:
from IPython.core.display import Markdown

Markdown(resposta["messages"][-1].text)

25 °C correspondem a **77 °F**. A conversão é \(F = \frac{9}{5}C + 32\): \(\frac{9}{5}\times25 + 32 = 77\).

### Inspecionando o ciclo do agente

O estado retornado contém as mensagens da interação. Mensagens do modelo podem incluir solicitações de chamada de ferramenta; mensagens de ferramenta contêm os resultados devolvidos pelas funções.

In [ ]:
for mensagem in resposta["messages"]:
    print("\nTipo:", mensagem.type)
    print("Conteúdo:", mensagem.content)
    if getattr(mensagem, "tool_calls", None):
        print("Chamadas de ferramentas:", mensagem.tool_calls)


Tipo: human
Conteúdo: Quantos graus Fahrenheit correspondem a 25 graus Celsius?

Tipo: ai
Conteúdo: [{'arguments': '{"celsius":25}', 'call_id': 'call_p5YESvXNKr3EqW73Gfsf64Mn', 'name': 'converter_celsius_fahrenheit', 'type': 'function_call', 'id': 'fc_07b7f8516bcc88e9006afb1f55b45487d2a3aa4cb071499083', 'status': 'completed'}]
Chamadas de ferramentas: [{'name': 'converter_celsius_fahrenheit', 'args': {'celsius': 25}, 'id': 'call_p5YESvXNKr3EqW73Gfsf64Mn', 'type': 'tool_call'}]

Tipo: tool
Conteúdo: 77.0

Tipo: ai
Conteúdo: [{'type': 'text', 'text': '25 °C correspondem a **77 °F**. A conversão é \\(F = \\frac{9}{5}C + 32\\): \\(\\frac{9}{5}\\times25 + 32 = 77\\).', 'annotations': [], 'id': 'msg_07b7f8516bcc88e9006afb1f56e1b087d298e3f6010c1a62bf', 'phase': 'final_answer'}]


## 5. Agente que combina várias ferramentas

Neste exemplo, o agente dispõe de ferramentas aritméticas. A solicitação exige mais de uma operação, e o agente deve decompor o problema e encadear os resultados.

In [ ]:
@tool
def soma(a: float, b: float) -> float:
    """Soma dois números."""
    return a + b

@tool
def multiplicacao(a: float, b: float) -> float:
    """Multiplica dois números."""
    return a * b

@tool
def potencia(base: float, expoente: float) -> float:
    """Calcula um número elevado a uma potência."""
    return base ** expoente

agente_calculadora = create_agent(
    model = modelo,
    tools = [soma, multiplicacao, potencia],
    system_prompt = "Resolva operações matemáticas usando as ferramentas disponíveis. Apresente a expressão e o resultado."
)

In [ ]:
consulta = "Calcule (3 elevado à quarta potência) multiplicado pela soma de 7 e 3. Depois, eleve o resultado ao quadrado."

resposta = agente_calculadora.invoke({
    "messages": [{"role": "user", "content": consulta}]
})

print(resposta["messages"][-1].content)

[{'type': 'text', 'text': '\\[\n\\bigl(3^4 \\times (7+3)\\bigr)^2\n= (81 \\times 10)^2\n= 810^2\n= \\boxed{656\\,100}\n\\]', 'annotations': [], 'id': 'msg_0e9bb81f37bdff56006afb1fd02d9c87d2bcab9f233efed457', 'phase': 'final_answer'}]


In [ ]:
from IPython.core.display import Markdown

Markdown(resposta["messages"][-1].text)

\[
\bigl(3^4 \times (7+3)\bigr)^2
= (81 \times 10)^2
= 810^2
= \boxed{656\,100}
\]

In [ ]:
import re
from IPython.display import display, Markdown

texto = resposta["messages"][-1].text

# Converter equações delimitadas por [ ... ] para $$ ... $$
texto = re.sub(
    r'\[\s*(\\.*?\\boxed\{.*?\}.*?)\s*\]',
    r'$$\1$$',
    texto,
    flags=re.DOTALL
)

display(Markdown(texto))

\$$\bigl(3^4 \times (7+3)\bigr)^2
= (81 \times 10)^2
= 810^2
= \boxed{656\,100}
\$$

## 6. Ferramentas com dados locais: catálogo de disciplinas

Agentes também podem selecionar ferramentas para consultar conjuntos de dados disponíveis no programa. O exemplo utiliza um pequeno catálogo fictício de disciplinas.

In [ ]:
disciplinas = [
    {"codigo": "BCM0505", "nome": "Processamento de Linguagem Natural", "creditos": 4, "area": "Computação"},
    {"codigo": "BCM0506", "nome": "Aprendizado de Máquina", "creditos": 4, "area": "Computação"},
    {"codigo": "MCTB010", "nome": "Estatística", "creditos": 3, "area": "Matemática"},
    {"codigo": "BCL030", "nome": "Introdução à Ciência de Dados", "creditos": 4, "area": "Computação"},
    {"codigo": "MCTA020", "nome": "Álgebra Linear", "creditos": 4, "area": "Matemática"}
]

@tool
def buscar_disciplina(termo: str) -> list[dict]:
    """Busca disciplinas cujo nome ou código contenha o termo informado."""
    termo = termo.lower()
    return [
        disciplina for disciplina in disciplinas
        if termo in disciplina["nome"].lower() or termo in disciplina["codigo"].lower()
    ]

@tool
def listar_por_area(area: str) -> list[dict]:
    """Lista as disciplinas de uma área, como Computação ou Matemática."""
    return [
        disciplina for disciplina in disciplinas
        if disciplina["area"].lower() == area.lower()
    ]

@tool
def contar_creditos(codigo: str) -> str:
    """Retorna a quantidade de créditos de uma disciplina a partir de seu código."""
    for disciplina in disciplinas:
        if disciplina["codigo"].lower() == codigo.lower():
            return f'{disciplina["nome"]}: {disciplina["creditos"]} créditos.'
    return "Disciplina não encontrada."

agente_disciplinas = create_agent(
    model = modelo,
    tools = [buscar_disciplina, listar_por_area, contar_creditos],
    system_prompt = (
        "Você é um assistente acadêmico. Consulte o catálogo usando as ferramentas. "
        "Não invente disciplinas ou informações que não estejam nos dados."
    )
)

In [ ]:
resposta = agente_disciplinas.invoke({
    "messages": [{
        "role": "user",
        "content": "Quais disciplinas são da área de Computação? Informe também os créditos de Processamento de Linguagem Natural."
    }]
})

print(resposta["messages"][-1].content)

[{'type': 'text', 'text': 'As disciplinas da área de Computação são:\n\n- **BCM0505 — Processamento de Linguagem Natural**\n- **BCM0506 — Aprendizado de Máquina**\n- **BCL030 — Introdução à Ciência de Dados**\n\n**Processamento de Linguagem Natural tem 4 créditos.**', 'annotations': [], 'id': 'msg_00871cf86daeacf0006afb208eb61c87d298a3a79f8d73f02a', 'phase': 'final_answer'}]


In [ ]:
from IPython.core.display import Markdown

Markdown(resposta["messages"][-1].text)

As disciplinas da área de Computação são:

- **BCM0505 — Processamento de Linguagem Natural**
- **BCM0506 — Aprendizado de Máquina**
- **BCL030 — Introdução à Ciência de Dados**

**Processamento de Linguagem Natural tem 4 créditos.**

**Discussão:** quais ferramentas foram necessárias? O agente poderia responder à pergunta usando apenas uma delas? O que aconteceria se o catálogo não contivesse a disciplina solicitada?

## 7. Agente para tarefas de PLN: análise textual

Neste exemplo, as ferramentas executam operações simples sobre textos: contagem de palavras, identificação de palavras mais frequentes e extração de trechos. São operações determinísticas realizadas por Python; o modelo decide quais ferramentas são pertinentes à solicitação.

In [ ]:
from collections import Counter
import re

@tool
def contar_palavras(texto: str) -> dict:
    """Conta palavras e caracteres de um texto. A contagem de palavras considera sequências alfanuméricas."""
    palavras = re.findall(r"\b[\wÀ-ÿ]+\b", texto.lower())
    return {
        "quantidade_palavras": len(palavras),
        "quantidade_caracteres": len(texto)
    }

@tool
def palavras_frequentes(texto: str, quantidade: int = 5) -> list[tuple[str, int]]:
    """Retorna as palavras mais frequentes do texto, desconsiderando uma pequena lista de palavras funcionais em português."""
    stopwords = {"a", "o", "as", "os", "de", "da", "do", "e", "em", "um", "uma", "para", "que", "com"}
    palavras = re.findall(r"\b[\wÀ-ÿ]+\b", texto.lower())
    frequencias = Counter(p for p in palavras if p not in stopwords)
    return frequencias.most_common(max(1, min(quantidade, 20)))

@tool
def localizar_termo(texto: str, termo: str) -> str:
    """Localiza ocorrências de um termo no texto e retorna as frases em que aparece."""
    frases = re.split(r"(?<=[.!?])\s+", texto)
    encontradas = [frase for frase in frases if termo.lower() in frase.lower()]
    return "\n".join(encontradas) if encontradas else "O termo não foi encontrado."

agente_pln = create_agent(
    model = modelo,
    tools = [contar_palavras, palavras_frequentes, localizar_termo],
    system_prompt = "Você é um assistente de PLN. Use as ferramentas para obter estatísticas textuais e diferencie resultados calculados de interpretações."
)

In [ ]:
texto = (
    "A inteligência artificial tem transformado diferentes atividades. "
    "Na educação, ferramentas de inteligência artificial podem apoiar estudantes e professores. "
    "Entretanto, o uso de inteligência artificial também exige avaliação crítica, transparência e responsabilidade."
)

resposta = agente_pln.invoke({
    "messages": [{
        "role": "user",
        "content": f"Analise este texto: informe quantidade de palavras, cinco termos mais frequentes e as frases que mencionam educação.\n\nTexto: {texto}"
    }]
})

print(resposta["messages"][-1].content)

[{'id': 'rs_0a8a655040162006006afb20f40e0887d28a2def93e9402536', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABq-yD3-Mq4xa3ARC2KN5WsRh5lyjjjyg74CfAwYTTorTPAK8AKnSRgK1q-dE8O3NdLwo9q40tAvOhbPx6joesfIUNT4jBrZaP4y-EEbCbAm81lnQSnsInUXzoOTOOdqdkWJ3tUomsx5PZ8draZeYK4gq_m__a1I9L7fCRiv6bdyWgcyn5S8nG_3tATZXLtoc4wwXwafv35tFBGOTc0S3V3ZOxUgeC6sPjEXERbYhKR4gBPvUeyATw_POaoAmlSenVYcYIIl-Z9EtYPhXtV-CDdTrmrWGoCH3CyONCli15NbhBggOo6pQ600YatThtBArt426csoShgJbcDOgifDn7PcmDGTDUDdnUKl4SdSZXuiSpQgFb-q1oSZuAlz16v-VcICXbw1roVSLGSJJ1NeI9cFb8vIAPjZKahTG7vwvy9RbwnKXr3nt1qSl7bZqQHoHimpyzwuCeWoLzOPn7AlmEexntrfgxOskRurK1ns_t0f-Vt9VNjJli11kOXcfmYz7FbSrsNTD1bHexsVfwbXxWFsFvxr2Je_hCMquaaX41WsOb8Zw0X3omBbsIKgRRV-g4N_rFIEFBlOvvZVje89IUXxkKFdeveQ3StkN0rlPmIVDQehqMlsTJE8DbE1fLXfNJ0z0UiChKP47CNdJMdLNJe9Si1IUFOoqykPkhHVqP7wLBRjJk-IDf_1fkHqC7YKqx2ZrYbKpJ1GGHxk6XdWKRXvejk-s7YzPeQLio3lB5sA1e2wvS0TVnp_L0ls46vK2qbAZUfMx8_rRowm6f44BPaLHplhNLaQm-uaeWh5TnlhFPi8TOgSTieRKEsL0SZ_XQPCAcA_qqaHq4uglbbog0v3W

In [ ]:
from IPython.core.display import Markdown

Markdown(resposta["messages"][-1].text)

**Resultados calculados**
- **Quantidade de palavras:** 31
- **Termos mais frequentes:** “inteligência” (3) e “artificial” (3). Para as três posições restantes, há empate: vários termos aparecem uma vez. Entre eles, “tem”, “transformado” e “diferentes”.
- **Frase que menciona educação:** “Na educação, ferramentas de inteligência artificial podem apoiar estudantes e professores.”

**Interpretação**
O texto apresenta a inteligência artificial como possível apoio a estudantes e professores, mas ressalta a importância de seu uso crítico, transparente e responsável.

## 8. Agente de análise de sentimentos com ferramenta especializada

O agente pode escolher uma ferramenta de PLN especializada. Para tornar o exemplo reproduzível sem depender de outro serviço, a ferramenta abaixo utiliza um léxico pequeno e explícito. **Não é um classificador robusto**: serve para demonstrar a arquitetura de integração entre agente e ferramenta.

In [ ]:
lexico_positivo = {"excelente", "ótimo", "ótima", "bom", "boa", "adorei", "rápido", "rápida", "recomendo", "maravilhoso"}
lexico_negativo = {"péssimo", "péssima", "ruim", "demorado", "demorada", "defeito", "problema", "odiei", "caro", "cara"}

@tool
def classificar_sentimento(texto: str) -> dict:
    """Classifica um texto com um léxico simples de palavras positivas e negativas. Retorna contagens e polaridade aproximada."""
    palavras = re.findall(r"\b[\wÀ-ÿ]+\b", texto.lower())
    positivas = [p for p in palavras if p in lexico_positivo]
    negativas = [p for p in palavras if p in lexico_negativo]
    if len(positivas) > len(negativas):
        sentimento = "positivo"
    elif len(negativas) > len(positivas):
        sentimento = "negativo"
    else:
        sentimento = "neutro ou inconclusivo"
    return {
        "sentimento": sentimento,
        "termos_positivos": positivas,
        "termos_negativos": negativas
    }

agente_sentimentos = create_agent(
    model = modelo,
    tools = [classificar_sentimento],
    system_prompt = (
        "Você auxilia na análise de avaliações. Use a ferramenta para obter a classificação lexical. "
        "Explique que a abordagem é simplificada e que ironia, contexto e negação podem afetar o resultado."
    )
)

In [ ]:
avaliacao = "O produto é ótimo e chegou rápido, mas achei caro."

resposta = agente_sentimentos.invoke({
    "messages": [{"role": "user", "content": f"Analise o sentimento desta avaliação: {avaliacao}"}]
})

print(resposta["messages"][-1].content)

[{'type': 'text', 'text': 'A avaliação tem sentimento **positivo**, com ressalvas: elogia o produto e a rapidez da entrega, mas considera o preço alto.\n\nA classificação é simplificada e baseada em palavras. Ironia, contexto e negação podem afetar o resultado.', 'annotations': [], 'id': 'msg_01929e13f02cb61b006afb21312edc87d2b6cd882110ba7e01', 'phase': 'final_answer'}]


In [ ]:
from IPython.core.display import Markdown

Markdown(resposta["messages"][-1].text)

A avaliação tem sentimento **positivo**, com ressalvas: elogia o produto e a rapidez da entrega, mas considera o preço alto.

A classificação é simplificada e baseada em palavras. Ironia, contexto e negação podem afetar o resultado.

**Experimento:** compare “O produto não é bom” com “O produto é bom”. A ferramenta lexical simples não compreende adequadamente a negação.

## 9. Agente que escolhe entre ferramentas de PLN

Agora o agente dispõe de ferramentas com finalidades diferentes. A descrição das ferramentas ajuda o modelo a decidir se deve traduzir, resumir ou contar palavras. A tradução e o resumo deste exemplo são simulados por regras fixas para demonstrar seleção de ferramentas; não substituem um sistema de tradução ou sumarização.

In [ ]:
@tool
def traduzir_termos(texto: str, idioma: str = "inglês") -> str:
    """Traduz algumas expressões predefinidas para inglês. Demonstração limitada, não é um tradutor geral."""
    mapa = {
        "inteligência artificial": "artificial intelligence",
        "processamento de linguagem natural": "natural language processing",
        "aprendizado de máquina": "machine learning",
        "redes neurais": "neural networks"
    }
    resultado = texto
    for origem, destino in mapa.items():
        resultado = re.sub(origem, destino, resultado, flags=re.IGNORECASE)
    return resultado

@tool
def resumir_por_extracao(texto: str, max_frases: int = 2) -> str:
    """Gera um resumo extrativo simples selecionando as primeiras frases do texto."""
    frases = re.split(r"(?<=[.!?])\s+", texto.strip())
    return " ".join(frases[:max(1, max_frases)])

agente_textual = create_agent(
    model = modelo,
    tools = [contar_palavras, traduzir_termos, resumir_por_extracao],
    system_prompt = (
        "Você é um assistente de processamento textual. Escolha a ferramenta adequada ao pedido. "
        "Informe limitações quando uma ferramenta executar apenas uma demonstração simplificada."
    )
)

In [ ]:
texto = (
    "O processamento de linguagem natural estuda métodos computacionais para analisar e gerar linguagem humana. "
    "Suas aplicações incluem tradução automática, análise de sentimentos e sistemas de perguntas e respostas."
)

resposta = agente_textual.invoke({
    "messages": [{
        "role": "user",
        "content": f"Faça um resumo em duas frases e informe a quantidade de palavras do texto: {texto}"
    }]
})

print(resposta["messages"][-1].content)

[{'type': 'text', 'text': 'O processamento de linguagem natural utiliza métodos computacionais para analisar e gerar linguagem humana. Entre suas aplicações estão tradução automática, análise de sentimentos e sistemas de perguntas e respostas.\n\n**Quantidade de palavras no texto original:** 28.', 'annotations': [], 'id': 'msg_05e53fb6f6d38e62006afb2188d2c887d2a082c95de5e67f68', 'phase': 'final_answer'}]


In [ ]:
from IPython.core.display import Markdown

Markdown(resposta["messages"][-1].text)

O processamento de linguagem natural utiliza métodos computacionais para analisar e gerar linguagem humana. Entre suas aplicações estão tradução automática, análise de sentimentos e sistemas de perguntas e respostas.

**Quantidade de palavras no texto original:** 28.

## 10. Saída estruturada com Pydantic

Além de texto livre, um agente pode retornar dados validados segundo um esquema. A API atual permite informar diretamente a classe Pydantic em `response_format`. O resultado validado fica disponível em `structured_response`.

Neste exemplo, o agente extrai informações de uma avaliação e retorna uma estrutura com nota, sentimento, aspectos mencionados e justificativa.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class AnaliseAvaliacao(BaseModel):
    nota: int | None = Field(description="Nota de 1 a 5, se estiver explícita no texto", ge=1, le=5)
    sentimento: Literal["positivo", "negativo", "misto", "neutro"]
    aspectos: list[str] = Field(description="Aspectos do produto ou serviço mencionados")
    justificativa: str = Field(description="Justificativa breve baseada no texto")

agente_estruturado = create_agent(
    model = modelo,
    tools = [],
    response_format = AnaliseAvaliacao,
    system_prompt = "Extraia as informações da avaliação. Não invente uma nota quando ela não estiver explícita."
)

In [ ]:
resultado = agente_estruturado.invoke({
    "messages": [{
        "role": "user",
        "content": "Avaliação: Dei 4 estrelas. O aplicativo é fácil de usar, mas apresenta falhas quando tento finalizar o pagamento."
    }]
})

analise = resultado["structured_response"]
analise

AnaliseAvaliacao(nota=4, sentimento='misto', aspectos=['Facilidade de uso do aplicativo', 'Falhas ao finalizar o pagamento'], justificativa='A avaliação elogia a facilidade de uso, mas relata problemas ao tentar concluir o pagamento.')

O objeto retornado pode ser utilizado diretamente em outros componentes Python, por exemplo para montar um DataFrame ou exportar resultados.

In [ ]:
import pandas as pd

df_analise = pd.DataFrame([analise.model_dump()])
display(df_analise)

,nota,sentimento,aspectos,justificativa
0,4,misto,"[Facilidade de uso do aplicativo, Falhas ao fi...","A avaliação elogia a facilidade de uso, mas re..."


## 11. Memória de conversação com `thread_id`

No exemplo original, o histórico era recuperado manualmente e concatenado à nova mensagem. Uma alternativa oferecida pela documentação atual é configurar um *checkpointer*. Com `InMemorySaver`, o estado da conversa fica armazenado em memória durante a execução do programa.

Para recuperar a conversa, reutilize o mesmo `thread_id`. Uma nova identificação inicia outra conversa.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from uuid import uuid4

memoria = InMemorySaver()

agente_com_memoria = create_agent(
    model = modelo,
    tools = [],
    checkpointer = memoria,
    system_prompt = "Você é um assistente que mantém o contexto da conversa."
)

config = {
    "configurable": {
        "thread_id": str(uuid4())
    }
}

In [ ]:
primeira = agente_com_memoria.invoke(
    {"messages": [{"role": "user", "content": "Meu nome é Marina e estou estudando processamento de linguagem natural."}]},
    config=config
)

print(primeira["messages"][-1].content)

Prazer, Marina! Vou levar em conta que você está estudando processamento de linguagem natural nesta conversa.


In [ ]:
segunda = agente_com_memoria.invoke(
    {"messages": [{"role": "user", "content": "Qual é meu nome e qual assunto estou estudando?"}]},
    config=config
)

print(segunda["messages"][-1].content)

Seu nome é Marina, e você está estudando processamento de linguagem natural.


**Teste:** execute a pergunta seguinte com outro `thread_id`. Compare o resultado. O `InMemorySaver` é adequado para demonstrações e testes; os dados não constituem armazenamento persistente entre reinicializações do ambiente.

## 12. Streaming: acompanhar a execução do agente

Em vez de aguardar a resposta completa, `stream` permite receber atualizações durante a execução. O modo `updates` apresenta atualizações de estado produzidas pelos passos do agente, o que é útil para observar a interação com ferramentas.

In [ ]:
@tool
def consultar_prazo(dias: int) -> str:
    """Calcula a data aproximada de vencimento em dias a partir de hoje. Use apenas para demonstração."""
    from datetime import date, timedelta
    data = date.today() + timedelta(days=dias)
    return data.strftime("%d/%m/%Y")

agente_stream = create_agent(
    model = modelo,
    tools = [consultar_prazo],
    system_prompt = "Responda de forma objetiva e utilize a ferramenta quando a pergunta envolver calcular uma data futura."
)

In [ ]:
for atualizacao in agente_stream.stream(
    {"messages": [{"role": "user", "content": "Qual será a data daqui a 30 dias?"}]},
    stream_mode="updates"
):
    print(atualizacao)

{'model': {'messages': [AIMessage(content=[{'arguments': '{"dias":30}', 'call_id': 'call_S1DpZiAi1hlMumkRCttA0dOl', 'name': 'consultar_prazo', 'type': 'function_call', 'id': 'fc_013dfb420cf2d0b9006afb2234a80487d2930c5cffb67c3384', 'status': 'completed'}], additional_kwargs={}, response_metadata={'id': 'resp_013dfb420cf2d0b9006afb223344b087d2b5da1d83550920ac', 'created_at': 1794843187.0, 'metadata': {}, 'model': 'gpt-6-luna', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'gpt-6-luna'}, id='resp_013dfb420cf2d0b9006afb223344b087d2b5da1d83550920ac', tool_calls=[{'name': 'consultar_prazo', 'args': {'dias': 30}, 'id': 'call_S1DpZiAi1hlMumkRCttA0dOl', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 93, 'output_tokens': 21, 'total_tokens': 114, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}, 'output_token_details': {'reasoning': 0}})]}}
{'tools': {'messages': [ToolMessage(content='1

## 13. Tratamento de entradas inválidas e limites das ferramentas

Ferramentas devem validar entradas e retornar mensagens compreensíveis. Não se deve pressupor que o modelo sempre fornecerá argumentos válidos.

In [ ]:
@tool
def calcular_media(valores: list[float]) -> str:
    """Calcula a média aritmética de uma lista de números. A lista não pode estar vazia."""
    if not valores:
        return "Erro: informe ao menos um valor."
    return f"Média: {sum(valores) / len(valores):.2f}"

agente_media = create_agent(
    model = modelo,
    tools = [calcular_media],
    system_prompt = "Ajude a calcular médias. Se faltarem valores, solicite os dados ao usuário."
)

In [ ]:
resposta = agente_media.invoke({
    "messages": [{"role": "user", "content": "Calcule a média dos valores 7, 8.5, 9 e 6.5."}]
})

print(resposta["messages"][-1].content)

[{'type': 'text', 'text': 'A média é **7,75**.', 'annotations': [], 'id': 'msg_0a24daac1a8c1733006afb228a043c87d29ca4d5aa3cc50f96', 'phase': 'final_answer'}]


In [ ]:
from IPython.core.display import Markdown

Markdown(resposta["messages"][-1].text)

A média é **7,75**.

## 14. Projeto integrador: assistente de análise de comentários

O projeto reúne diferentes elementos apresentados no notebook. O agente recebe comentários de usuários e pode:
- calcular estatísticas básicas;
- classificar sentimento por meio de um léxico didático;
- localizar termos ou frases;
- produzir uma síntese textual.

O conjunto de comentários é pequeno e está definido diretamente no código.

In [ ]:
comentarios = [
    "O atendimento foi excelente e a entrega foi rápida.",
    "O produto apresentou defeito e o suporte demorou para responder.",
    "Gostei da qualidade, mas o preço é muito alto.",
    "A experiência foi boa e recomendo a loja.",
    "O pedido chegou atrasado e a embalagem estava danificada."
]

@tool
def estatisticas_comentarios() -> dict:
    """Retorna a quantidade de comentários e o total de palavras no conjunto de comentários."""
    total_palavras = sum(
        len(re.findall(r"\b[\wÀ-ÿ]+\b", comentario.lower()))
        for comentario in comentarios
    )
    return {"quantidade_comentarios": len(comentarios), "total_palavras": total_palavras}

@tool
def sentimento_comentarios() -> list[dict]:
    """Classifica cada comentário com o léxico simples de palavras positivas e negativas definido no notebook."""
    resultados = []
    for comentario in comentarios:
        palavras = re.findall(r"\b[\wÀ-ÿ]+\b", comentario.lower())
        positivas = sum(p in lexico_positivo for p in palavras)
        negativas = sum(p in lexico_negativo for p in palavras)
        if positivas > negativas:
            classe = "positivo"
        elif negativas > positivas:
            classe = "negativo"
        else:
            classe = "neutro ou misto"
        resultados.append({"comentario": comentario, "sentimento": classe})
    return resultados

@tool
def buscar_comentarios(termo: str) -> list[str]:
    """Retorna os comentários que contêm o termo informado."""
    return [c for c in comentarios if termo.lower() in c.lower()]

agente_projeto = create_agent(
    model = modelo,
    tools = [estatisticas_comentarios, sentimento_comentarios, buscar_comentarios],
    system_prompt = (
        "Você é um analista de comentários de clientes. Use as ferramentas para fundamentar a análise. "
        "Deixe claro que a classificação de sentimento é lexical e simplificada."
    )
)

In [ ]:
resposta = agente_projeto.invoke({
    "messages": [{
        "role": "user",
        "content": "Analise os comentários: informe quantos existem, apresente a classificação de sentimento de cada um e destaque os comentários que mencionam entrega."
    }]
})

print(resposta["messages"][-1].content)

[{'id': 'rs_0e09c4dd0298c97b006afb22c73d1087d2af4c26dfc05a44ce', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABq-yLKQqhYtpopP1RBLVqYOTlPrn24Tc9aMk2t5nhBiEtpkUGc27KoT13_3xdIn_TAV72mhmwM_r5R-9wVXPjMCtNkW71bXHj2_FRlAnYsjdJ3e44owj9TtOOjPPrihPP1S-FJYslXXql41j9RQH1Uwtbu0IEBZI6tS7t2jpbFlVFPyJRZmBgx9EwB2vNKwgtoXdHAdH-mwTn-IaiiVraJ-st5rScKzvoIRKyLcORL8hTCvuePaCj6D6pZ0vikxUXzXeoz1kbnnyG-imqiJVvIdPHfr8xSlIbQbvhTmnhq68o9B9FPJXYy8efDkpfXfwFztqyIOSdOUMptEU-4bgH_N5Q4pjiVerofUOkf6hS5bpS5jmgNGYlOhZvX988FhrxIrrSWMRt93ocApMKLfsSmhm3aBezjv0ly8h3NE4HcEtLPnhkG71B2j9gBf5slc5o_kfHFfjRr9fGeAf_efL2T18fmKzeryS6nWICUn0YpXkshqPDraVYSDw1V_yRIGbv-FtgU_AIE3-4Eg_KH2STd1XlzG72NK4KQwKhfdeDAm7o5kAhsSZ-Z35vNeT7KnE20QBaQY7J4zNLhNX8uVVud5Z9B7sqEcCPDqEDzhoJlHFUgAhDnWeEWjomVWucIz8WlngvcO19gDVG_bmw3aNLm72Uzr10kol2bzbFZXFpL2m5OqIxQ_j53NjiIDDLWvcYmdjMLBMBeVIP01xAjxY6w61FdS-TeM4GUHsT4vATlai3dQoJZbIUVDQ4mYpc4nk9iWt7D38WhUf8fokWfn_uXA7iu1jsUEhzsaRUTxLfhesdDO3T9zBg4paltUotspUmn6UgP45nTVG7AE_3uamaeQq

In [ ]:
from IPython.core.display import Markdown

Markdown(resposta["messages"][-1].text)

Foram analisados **5 comentários**. A classificação de sentimento abaixo é **lexical e simplificada**: baseia-se em um léxico de palavras positivas e negativas e pode não captar plenamente o contexto.

| # | Comentário | Sentimento |
|---|---|---|
| 1 | “O atendimento foi excelente e a entrega foi rápida.” | Positivo |
| 2 | “O produto apresentou defeito e o suporte demorou para responder.” | Negativo |
| 3 | “Gostei da qualidade, mas o preço é muito alto.” | Neutro ou misto |
| 4 | “A experiência foi boa e recomendo a loja.” | Positivo |
| 5 | “O pedido chegou atrasado e a embalagem estava danificada.” | Neutro ou misto |

**Comentários relacionados à entrega:**
- “O atendimento foi excelente e a entrega foi rápida.” — menciona explicitamente “entrega”.
- “O pedido chegou atrasado e a embalagem estava danificada.” — relata atraso na chegada do pedido.

## 15. Síntese: componentes fundamentais

| Componente | Função |
|---|---|
| `ChatOpenAI` | Define o modelo de linguagem |
| `@tool` | Expõe uma função Python como ferramenta |
| `create_agent` | Cria o agente a partir do modelo, ferramentas e instruções |
| `system_prompt` | Define o comportamento esperado do agente |
| `invoke` | Executa o agente e retorna o estado final |
| `messages` | Mantém o histórico de mensagens da execução |
| `response_format` | Define o formato de saída estruturada |
| `InMemorySaver` | Armazena estado de conversação em memória |
| `thread_id` | Identifica uma conversa para recuperar seu estado |
| `stream` | Permite acompanhar atualizações durante a execução |

A documentação atual descreve o agente como um modelo que utiliza ferramentas em um ciclo até concluir a tarefa. O comportamento é moldado pelo modelo, pelas ferramentas, pelo prompt e, em cenários mais avançados, por middleware.

**Documentação:** [LangChain — Agents](https://docs.langchain.com/oss/python/langchain/agents).